# Replay cleaning walkthrough: evidence, affected rows, and fixes

This notebook uses **pandas and PyArrow**, with no Spark session or Java required.
It demonstrates eight replay-cleaning issues across completed matches in `data/parsed`.

For each issue:
1. Read the explanation and evidence.
2. Inspect the affected raw rows.
3. Run the fix or validation cell and check its assertions.

Issues 1-6 demonstrate transformations in memory. Issues 7-8 inspect the new per-tick
`kda_changes` input and the saved counter-based outputs in **`data/output/clean_kda`**.
They compare the old event-counting approach with credited totals, then independently
check those totals against the final raw scoreboard. The attribution solver itself
runs in the [Spark cleaner](../spark_jobs/clean_replays.py), not in this notebook.

**This notebook only reads files.** It does not re-parse replays, run Spark, change raw
values, write clean tables, or load the warehouse. Exact final K/D/A totals do not prove
every individual assist-to-death link; assumption labels and same-side credit stay visible.

Install `python -m pip install -r notebooks/requirements.txt` from the repository root,
select that environment, then run cells in order. Both the repository root and `notebooks/`
work as starting directories. Re-parse old matches and run the new cleaner first if
`kda_changes` or completed `clean_kda` outputs are missing.

References: [raw contract](../docs/data-contract.md),
[counter-based job and output additions](../docs/running-kda-clean-job.md),
[base clean contract](../docs/clean-contract.md). The clean contract, loader and dbt models support the new credit columns. The
dashboard uses credited kills and eligible team-kill assists for participation.


In [27]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "spark_jobs" / "replay_spec.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook from the dota-pipeline project folder.")
PARSED = ROOT / "data" / "parsed"
CLEAN_KDA = ROOT / "data" / "output" / "clean_kda"
FOCUS_MATCH_ID = None  # Set a completed match ID, or leave None to use the first.
NULL_HANDLE = 16777215
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
TABLE_NAMES = ["match", "players", "combatlog", "names", "entity_names", "gamerules",
               "heroes", "player_resource", "kda_changes", "team_data", "wards", "item_entities"]

# Read physical files directly: partition columns may only exist in their folder names.
# Nullable pandas dtypes preserve absent integers and booleans instead of filling them with zero.
def read_table(name, match_id):
    partition = PARSED / name / f"match_id={match_id}"
    files = sorted(partition.glob("*.parquet"))
    if not files:
        if name == "wards":
            return None
        if name == "kda_changes":
            raise FileNotFoundError(
                f"Missing per-tick counters: {partition}. Re-parse with the updated parser; "
                "one-second player_resource snapshots cannot replace kda_changes.")
        raise FileNotFoundError(f"No Parquet files: {partition}")
    df = pd.concat([pq.ParquetFile(f).read().to_pandas().convert_dtypes()
                    for f in files], ignore_index=True)
    if "match_id" in df and not df["match_id"].eq(match_id).all():
        raise ValueError(f"Embedded match_id disagrees with partition: {partition}")
    df["match_id"] = pd.Series(match_id, index=df.index, dtype="Int64")
    return df

# Null group keys are retained so audits do not silently discard missing values.
def grouped_counts(df, keys, label="rows"):
    return df.groupby(keys, dropna=False).size().rename(label).reset_index()

def duplicate_groups(df, keys):
    counts = grouped_counts(df, keys)
    return counts.loc[counts["rows"] > 1]

def last_rows(df, keys, time="tick", tie_breakers=()):
    # Highest tick first, lowest handle on a tied tick when requested.
    order = [*keys, time, *tie_breakers]
    return df.sort_values(order, ascending=[True] * len(keys) + [False] + [True] * len(tie_breakers),
                          kind="stable").drop_duplicates(keys)

def true_mask(series):
    return series.eq(True).fillna(False)

def false_mask(series):
    # Missing flags remain unknown; report exclusions rather than assuming False.
    return series.eq(False).fillna(False)

In [28]:
REPLAY_FILES = sorted({*list((ROOT / "data").glob("*.dem")),
                       *list((ROOT / "data" / "output").glob("*.dem"))})
REPLAY_MATCH_IDS = sorted({int(p.stem) for p in REPLAY_FILES if p.stem.isdigit()})
marker_root = PARSED / "_complete"
COMPLETE_MATCH_IDS = sorted(int(p.name.split("=", 1)[1]) for p in marker_root.glob("match_id=*")
    if p.name.split("=", 1)[1].isdigit() and any(p.glob("*.parquet")))
LOCAL_MATCH_IDS = sorted(set(REPLAY_MATCH_IDS) | set(COMPLETE_MATCH_IDS))
PENDING_MATCH_IDS = sorted(set(REPLAY_MATCH_IDS) - set(COMPLETE_MATCH_IDS))
print(f"Coverage: {len(COMPLETE_MATCH_IDS)}/{len(LOCAL_MATCH_IDS)} discovered matches fully parsed")
print("Ready:", COMPLETE_MATCH_IDS)
print("Unparsed local replays:", PENDING_MATCH_IDS)
print("Local .dem files in data/ or data/output/:", len(REPLAY_MATCH_IDS))
if not COMPLETE_MATCH_IDS:
    raise RuntimeError(f"No completion markers found under {marker_root}. Check PARSED or parse replays first.")
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else COMPLETE_MATCH_IDS[0]
if MATCH_ID not in COMPLETE_MATCH_IDS:
    raise ValueError(f"Focus match {MATCH_ID} is not completed.")
print("Detailed inspection match:", MATCH_ID)

batch = {}
coverage_rows = []
for name in TABLE_NAMES:
    parts = []
    for match_id in COMPLETE_MATCH_IDS:
        df = read_table(name, match_id)
        coverage_rows.append({"match_id": match_id, "table": name,
                              "rows": 0 if df is None else len(df), "partition_present": df is not None})
        if df is not None:
            parts.append(df)
    if parts:
        batch[name] = pd.concat(parts, ignore_index=True)
coverage = pd.DataFrame({"match_id": pd.Series(COMPLETE_MATCH_IDS, dtype="Int64")})
coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report.pivot(index="match_id", columns="table", values="rows"))
print("Total loaded memory (MB):", round(sum(df.memory_usage(deep=True).sum()
                                          for df in batch.values()) / 1024**2, 1))

# Store one metric per match for a readable final summary.
metrics = {}
def report_counts(df, label):
    counts = grouped_counts(df, ["match_id"], label)
    result = coverage.merge(counts, on="match_id", how="left", validate="one_to_one")
    result[label] = result[label].fillna(0).astype("Int64")
    metrics[label] = result.set_index("match_id")[label]
    display(result)
    return result

def resolve_name(df, index_column, output_column):
    # Name indexes are local to each match. Null lookup keys cannot match.
    lookup = batch["names"][["match_id", "idx", "name"]].dropna(subset=["idx"]).rename(
        columns={"idx": index_column, "name": output_column})
    return df.merge(lookup, on=["match_id", index_column], how="left", validate="many_to_one")

Coverage: 10/10 discovered matches fully parsed
Ready: [9079737909, 9079737994, 9079738443, 9079738574, 9079740225, 9079740632, 9079741152, 9079743269, 9079745340, 9079746862]
Unparsed local replays: []
Local .dem files in data/ or data/output/: 10
Detailed inspection match: 9079737909


table,combatlog,entity_names,gamerules,heroes,item_entities,kda_changes,match,names,player_resource,players,team_data,wards
match_id,,,,,,,,,,,,
9079737909,21908,564,1508,14570,621,117,1,242,15080,10,15080,0
9079737994,31712,607,1348,14238,608,196,1,298,13480,10,13480,6
9079738443,14679,547,918,8154,342,43,1,218,9180,10,9180,10
9079738574,23895,605,1379,17525,1173,114,1,336,13790,10,13790,57
9079740225,6132,536,545,4320,185,49,1,155,5450,10,5450,5
9079740632,19802,579,1503,15356,439,247,1,271,15030,10,15030,6
9079741152,14863,581,1199,10740,443,96,1,291,12723,10,11990,17
9079743269,6613,555,761,24360,302,66,1,200,7610,10,7610,14
9079745340,10265,557,1079,10551,557,58,1,241,10790,10,10790,34


Total loaded memory (MB): 189.6


In [29]:
players_raw = batch["players"]
heroes_raw = batch["heroes"]
resource_raw = batch["player_resource"]
combat_raw = batch["combatlog"]
results = []

def record(issue, status, detail):
    results.append({"issue": issue, "status": status, "detail": detail})

def by_match(df, label="rows"):
    counts = grouped_counts(df, ["match_id"], label)
    out = coverage.merge(counts, on="match_id", how="left", validate="one_to_one")
    out[label] = out[label].fillna(0).astype("Int64")
    return out

def check_unique(df, key):
    assert not df[key].isna().any(axis=None), f"Null key: {key}"
    assert not df.duplicated(key).any(), f"Repeated key: {key}"

# Header names define the player roster within each match.
check_unique(players_raw, ["match_id", "slot"])
check_unique(players_raw, ["match_id", "hero_name"])

## Issue 1. Player identifiers use different numbering

**Why transform:** hero IDs such as 0, 2, ..., 18 cannot be joined directly to header slots 0-9.
Assist participant IDs are already in the 0-9 range. A universal divide-by-two rule is incorrect.

**Evidence cell:** compare raw hero identifiers with header slots, hero names, and teams. The
identity examples show why IDs differ; fractional assist slots demonstrate a wrong conversion.

In [30]:
id_examples = heroes_raw[["match_id", "class", "m_iPlayerID", "m_iTeamNum"]].drop_duplicates().copy()
id_examples["candidate_player_slot"] = id_examples["m_iPlayerID"] / 2
id_examples = id_examples.merge(players_raw[["match_id", "slot", "hero_name", "game_team"]],
    left_on=["match_id", "candidate_player_slot"], right_on=["match_id", "slot"],
    how="left", validate="many_to_one")
display(id_examples.sort_values(["match_id", "candidate_player_slot"]).head(12))
assist_id_examples = combat_raw["assist_players"].dropna().str.split(";").explode().str.strip()
assist_id_examples = pd.to_numeric(assist_id_examples[assist_id_examples.str.fullmatch(r"[0-9]+")]).drop_duplicates()
display(pd.DataFrame({"raw_assist_id": assist_id_examples,
    "wrong_if_divided_by_two": assist_id_examples / 2}).sort_values("raw_assist_id"))
print("Rows with raw hero IDs different from the required slot:",
      int(id_examples["m_iPlayerID"].ne(id_examples["candidate_player_slot"]).sum()))

,match_id,class,m_iPlayerID,m_iTeamNum,candidate_player_slot,slot,hero_name,game_team
3,9079737909,CDOTA_Unit_Hero_Juggernaut,0,2,0.0,0,npc_dota_hero_juggernaut,2
4,9079737909,CDOTA_Unit_Hero_Legion_Commander,2,2,1.0,1,npc_dota_hero_legion_commander,2
5,9079737909,CDOTA_Unit_Hero_Tidehunter,4,2,2.0,2,npc_dota_hero_tidehunter,2
6,9079737909,CDOTA_Unit_Hero_Axe,6,2,3.0,3,npc_dota_hero_axe,2
7,9079737909,CDOTA_Unit_Hero_Sniper,8,2,4.0,4,npc_dota_hero_sniper,2
8,9079737909,CDOTA_Unit_Hero_DrowRanger,10,3,5.0,5,npc_dota_hero_drow_ranger,3
9,9079737909,CDOTA_Unit_Hero_Sven,12,3,6.0,6,npc_dota_hero_sven,3
0,9079737909,CDOTA_Unit_Hero_Centaur,14,3,7.0,7,npc_dota_hero_centaur,3
1,9079737909,CDOTA_Unit_Hero_Huskar,16,3,8.0,8,npc_dota_hero_huskar,3
2,9079737909,CDOTA_Unit_Hero_Lina,18,3,9.0,9,npc_dota_hero_lina,3


,raw_assist_id,wrong_if_divided_by_two
3050,0,0.0
744,1,0.5
760,2,1.0
3070,3,1.5
838,4,2.0
771,5,2.5
203,6,3.0
863,7,3.5
785,8,4.0
762,9,4.5


Rows with raw hero IDs different from the required slot: 90


**Example: the same player has different identifiers in different tables**

In match `9079737909`:

```text
Hero: Lina
heroes.m_iPlayerID: 18
players.slot: 9
Assist participant ID for this player: 9
```

Joining hero ID `18` directly to header slot `9` would miss this player. For hero snapshots,
`18 / 2 = 9` gives the expected slot. Applying the same conversion to assist ID `9` would instead
produce `4.5`, which is not a player slot. Convert the identifier according to its source table.

**Fix:** convert validated hero IDs only, check the resulting team against the match roster, and
retain original rows that cannot be mapped. Do not divide assist IDs. The raw hero property is
kept in this intermediate frame so the conversion remains easy to inspect.

In [31]:
valid_hero_id = (heroes_raw["m_iPlayerID"].notna()
    & heroes_raw["m_iPlayerID"].mod(2).eq(0).fillna(False)
    & heroes_raw["m_iPlayerID"].between(0, 18).fillna(False))
hero_ids_for_review = heroes_raw.loc[~valid_hero_id].copy()
canonical_heroes = heroes_raw.loc[valid_hero_id].copy()
canonical_heroes["player_slot"] = (canonical_heroes["m_iPlayerID"] // 2).astype("Int64")
canonical_heroes = canonical_heroes.merge(players_raw[["match_id", "slot", "hero_name", "game_team"]].rename(
    columns={"slot": "player_slot"}), on=["match_id", "player_slot"], how="left", validate="many_to_one")
bad_team = canonical_heroes["hero_name"].isna() | canonical_heroes["m_iTeamNum"].ne(
    canonical_heroes["game_team"]).fillna(True)
hero_ids_for_review = pd.concat([hero_ids_for_review, canonical_heroes.loc[bad_team]], ignore_index=True)
canonical_heroes = canonical_heroes.loc[~bad_team].copy()
assert canonical_heroes["player_slot"].between(0, 9).all()
assert len(canonical_heroes) + len(hero_ids_for_review) == len(heroes_raw)
display(canonical_heroes[["match_id", "m_iPlayerID", "player_slot", "hero_name"]].drop_duplicates().head(12))
print("Unmapped hero rows retained for review:", len(hero_ids_for_review))
record("1. Hero player IDs", "Validated on loaded matches", f"{len(hero_ids_for_review)} hero rows retained for review")

,match_id,m_iPlayerID,player_slot,hero_name
0,9079737909,14,7,npc_dota_hero_centaur
1,9079737909,16,8,npc_dota_hero_huskar
2,9079737909,18,9,npc_dota_hero_lina
3,9079737909,0,0,npc_dota_hero_juggernaut
4,9079737909,2,1,npc_dota_hero_legion_commander
5,9079737909,4,2,npc_dota_hero_tidehunter
6,9079737909,6,3,npc_dota_hero_axe
7,9079737909,8,4,npc_dota_hero_sniper
8,9079737909,10,5,npc_dota_hero_drow_ranger
9,9079737909,12,6,npc_dota_hero_sven


Unmapped hero rows retained for review: 0


## Issue 2. Player-resource rows outside the roster

**Why filter:** only Radiant/Dire players with a valid team slot can become player statistics.
Rows for other teams or missing slots cannot be assigned to a player by guessing.

**Evidence cell:** show each invalid mapping and examples with ticks. The earlier audit found
733 rows at array index 10, team 1, with no team slot in match 9079741152. This is an extra array
entry, not evidence that 733 real players are missing.

In [32]:
resource = resource_raw.rename(columns={"m_vecPlayerData.m_iPlayerTeam": "team",
    "m_vecPlayerTeamData.m_iTeamSlot": "team_slot",
    "m_vecPlayerTeamData.m_nSelectedHeroID": "hero_id"}).copy()
valid_resource_mapping = resource["team"].isin([2, 3]) & resource["team_slot"].between(0, 4).fillna(False)
invalid_resource_rows = resource.loc[~valid_resource_mapping].copy()
display(by_match(invalid_resource_rows, "unmapped_rows"))
display(invalid_resource_rows[["match_id", "tick", "index", "team", "team_slot", "hero_id"]].head(10))
display(invalid_resource_rows.groupby(["match_id", "index", "team", "team_slot"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index())

,match_id,unmapped_rows
0,9079737909,0
1,9079737994,0
2,9079738443,0
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,733
7,9079743269,0
8,9079745340,0
9,9079746862,0


,match_id,tick,index,team,team_slot,hero_id
76680,9079741152,14010,10,1,<NA>,<NA>
76691,9079741152,14040,10,1,<NA>,<NA>
76702,9079741152,14070,10,1,<NA>,<NA>
76713,9079741152,14100,10,1,<NA>,<NA>
76724,9079741152,14130,10,1,<NA>,<NA>
76735,9079741152,14160,10,1,<NA>,<NA>
76746,9079741152,14190,10,1,<NA>,<NA>
76757,9079741152,14220,10,1,<NA>,<NA>
76768,9079741152,14250,10,1,<NA>,<NA>
76779,9079741152,14280,10,1,<NA>,<NA>


,match_id,index,team,team_slot,rows,first_tick,last_tick
0,9079741152,10,1,<NA>,733,14010,35970


**Example: an array entry does not identify a Radiant or Dire player**

```text
Match: 9079741152
Tick: 14010
Player-resource array index: 10
Team: 1
Team slot: <NA>
Hero ID: <NA>
```

This entry appears in **733 snapshots**, from tick `14010` through `35970`. These are repeated
observations of one array entry, not 733 different players. The clean player roster uses teams
`2` and `3`, each with team slots `0` to `4`. Team `1` with no team slot cannot be mapped using
that rule. Exclude these rows from player statistics and retain them for inspection; do not
replace the missing team slot with `0` or guess which player they represent.

In [33]:
# Inpect match_id 9079741152
one_match = resource[resource["match_id"] == 9079741152]
display(one_match.groupby(["match_id", "index", "team", "team_slot"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index())

,match_id,index,team,team_slot,rows,first_tick,last_tick
0,9079741152,0,2,0,1199,30,35970
1,9079741152,1,2,1,1199,30,35970
2,9079741152,2,2,2,1199,30,35970
3,9079741152,3,2,3,1199,30,35970
4,9079741152,4,2,4,1199,30,35970
5,9079741152,5,3,0,1199,30,35970
6,9079741152,6,3,1,1199,30,35970
7,9079741152,7,3,2,1199,30,35970
8,9079741152,8,3,3,1199,30,35970
9,9079741152,9,3,4,1199,30,35970


**Fix:** retain excluded rows separately, filter to teams 2/3 and slots 0-4, derive `player_slot`,
and check against the header roster. This follows the contract's instruction to skip non-playing
teams and avoids filling a missing team slot with zero.

In [34]:
resource_mapped = resource.loc[valid_resource_mapping].copy()
resource_mapped["player_slot"] = ((resource_mapped["team"] - 2) * 5 + resource_mapped["team_slot"]).astype("Int64")
roster = players_raw[["match_id", "slot", "game_team"]].rename(columns={"slot": "player_slot", "game_team": "header_team"})
resource_mapped = resource_mapped.merge(roster, on=["match_id", "player_slot"], how="left", validate="many_to_one")
assert resource_mapped["header_team"].notna().all()
assert resource_mapped["header_team"].eq(resource_mapped["team"]).all()
assert len(resource_mapped) + len(invalid_resource_rows) == len(resource_raw)
check_unique(resource_mapped, ["match_id", "player_slot", "tick"])
display(resource_mapped[["match_id", "tick", "index", "team", "team_slot", "player_slot"]].head(10))
print("Retained rows:", len(resource_mapped), "| Excluded rows kept for review:", len(invalid_resource_rows))
record("2. Non-roster resource rows", "Filtered with rows retained for review", f"{len(invalid_resource_rows)} rows excluded from player stats")

,match_id,tick,index,team,team_slot,player_slot
0,9079737909,30,0,2,0,0
1,9079737909,30,1,2,1,1
2,9079737909,30,2,2,2,2
3,9079737909,30,3,2,3,3
4,9079737909,30,4,2,4,4
5,9079737909,30,5,3,0,5
6,9079737909,30,6,3,1,6
7,9079737909,30,7,3,2,7
8,9079737909,30,8,3,3,8
9,9079737909,30,9,3,4,9


Retained rows: 113000 | Excluded rows kept for review: 733


## Issue 3. Hero selection is temporarily unset or changes

**Why transform:** a first snapshot can have hero ID 0; choosing the largest ID can select a
transient value such as 1000000. Neither is a reliable player-level hero assignment.

**Evidence cell:** show each player's observed hero-ID history and raw rows that disagree with
the final selection. A differing earlier value is historical state, not a value to overwrite in raw.

In [35]:
hero_history = resource_mapped.groupby(["match_id", "player_slot", "hero_id"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index()
changed_players = hero_history.groupby(["match_id", "player_slot"], dropna=False).size().rename("observed_ids").reset_index()
changed_players = changed_players.loc[changed_players["observed_ids"] > 1]
display(hero_history.merge(changed_players[["match_id", "player_slot"]], on=["match_id", "player_slot"]).head(20))
final_selection = last_rows(resource_mapped, ["match_id", "player_slot"])[["match_id", "player_slot", "tick", "hero_id"]].rename(
    columns={"tick": "selection_tick", "hero_id": "final_hero_id"})
hero_discrepancy = resource_mapped.merge(final_selection, on=["match_id", "player_slot"], validate="many_to_one")
hero_discrepancy = hero_discrepancy.loc[hero_discrepancy["hero_id"].ne(hero_discrepancy["final_hero_id"]).fillna(True)]
display(hero_discrepancy[["match_id", "tick", "player_slot", "hero_id", "final_hero_id"]].head(12))

,match_id,player_slot,hero_id,rows,first_tick,last_tick
0,9079737909,1,0,2,30,60
1,9079737909,1,104,1506,90,45240
2,9079737909,2,0,5,30,150
3,9079737909,2,29,1503,180,45240
4,9079737909,4,0,2,30,60
5,9079737909,4,35,1506,90,45240
6,9079737909,5,0,2,30,60
7,9079737909,5,6,1506,90,45240
8,9079737909,6,0,4,30,120
9,9079737909,6,18,1503,180,45240


,match_id,tick,player_slot,hero_id,final_hero_id
1,9079737909,30,1,0,104
2,9079737909,30,2,0,29
4,9079737909,30,4,0,35
5,9079737909,30,5,0,6
6,9079737909,30,6,0,18
7,9079737909,30,7,0,96
11,9079737909,60,1,0,104
12,9079737909,60,2,0,29
14,9079737909,60,4,0,35
15,9079737909,60,5,0,6


**Example: the first and largest hero IDs are both misleading**

In match `9079737909`, the player at clean slot `6` has this selection history:

```text
Ticks 30-120: hero_id = 0
Tick 150:    hero_id = 1000000
Tick 180 onward, through the final snapshot: hero_id = 18
Header hero name: npc_dota_hero_sven
```

Selecting the first observed ID would return `0`; taking the maximum would return `1000000`.
Neither describes the final assignment in these samples. The fix below selects the final mapped
snapshot for the player-level table, which gives `18` here. Earlier rows stay unchanged in the raw
data. This is a demonstrated selection rule, not proof that a final snapshot covers every game mode.

**Fix for this walkthrough:** use the final mapped snapshot for the one-row-per-player table.
Convert an absent or nonpositive final selection to null, retaining it for review. Do not impose
an invented upper bound on hero IDs. Final selection works for these samples; agree this rule
with the contract owner before using it for matches with hero replacements or unusual modes.

In [36]:
selected = final_selection.copy()
selected["hero_id"] = selected["final_hero_id"].where(selected["final_hero_id"].gt(0)).astype("Int64")
hero_selection_for_review = selected.loc[selected["hero_id"].isna()].copy()
clean_players = players_raw.rename(columns={"slot": "player_slot", "game_team": "team"}).merge(
    selected[["match_id", "player_slot", "hero_id"]], on=["match_id", "player_slot"], how="left", validate="one_to_one")
clean_players["team_slot"] = (clean_players["player_slot"] % 5).astype("Int64")
check_unique(clean_players, ["match_id", "player_slot"])
assert len(clean_players) == len(players_raw)
display(clean_players.head(12))
print("Final selections needing review:", len(hero_selection_for_review))
record("3. Stable hero assignment", "Sample rule demonstrated", f"Final snapshot selected; {len(hero_selection_for_review)} missing final IDs")

,player_slot,hero_name,team,match_id,hero_id,team_slot
0,0,npc_dota_hero_juggernaut,2,9079737909,8,0
1,1,npc_dota_hero_legion_commander,2,9079737909,104,1
2,2,npc_dota_hero_tidehunter,2,9079737909,29,2
3,3,npc_dota_hero_axe,2,9079737909,2,3
4,4,npc_dota_hero_sniper,2,9079737909,35,4
5,5,npc_dota_hero_drow_ranger,3,9079737909,6,0
6,6,npc_dota_hero_sven,3,9079737909,18,1
7,7,npc_dota_hero_centaur,3,9079737909,96,2
8,8,npc_dota_hero_huskar,3,9079737909,59,3
9,9,npc_dota_hero_lina,3,9079737909,25,4


Final selections needing review: 0


## Issue 4. Replay ticks do not equal seconds on the game clock

**Why transform:** replays include pregame and postgame. A chart using `tick / 30` puts the horn
at a positive time rather than 0:00 and can overstate duration.

**Evidence cell:** show each match's first in-progress tick and actual early rows beside the
incorrect clock. Negative pregame values are valid and must not be clipped to zero.

In [37]:
game = batch["gamerules"][["match_id", "tick", "m_pGameRules.m_nGameState",
    "m_pGameRules.m_flGameStartTime", "m_pGameRules.m_flGameEndTime"]].rename(columns={
    "m_pGameRules.m_nGameState": "state", "m_pGameRules.m_flGameStartTime": "start_time",
    "m_pGameRules.m_flGameEndTime": "end_time"})
starts = game.loc[game["state"].eq(5)].groupby("match_id")["tick"].min().rename("game_start_tick").reset_index()
display(starts)
clock_evidence = game.merge(starts, on="match_id", how="left", validate="many_to_one")
clock_evidence["incorrect_tick_divided_by_30"] = clock_evidence["tick"] / 30.0
display(clock_evidence.loc[clock_evidence["tick"] < clock_evidence["game_start_tick"]].groupby("match_id").head(2).head(12))

,match_id,game_start_tick
0,9079737909,4260
1,9079737994,6150
2,9079738443,7290
3,9079738574,6240
4,9079740225,6120
5,9079740632,5580
6,9079741152,6480
7,9079743269,6030
8,9079745340,6060
9,9079746862,7110


,match_id,tick,state,start_time,end_time,game_start_tick,incorrect_tick_divided_by_30
0,9079737909,30,2,0.0,0.0,4260,1.0
1,9079737909,60,2,0.0,0.0,4260,2.0
1508,9079737994,30,2,0.0,0.0,6150,1.0
1509,9079737994,60,2,0.0,0.0,6150,2.0
2856,9079738443,30,2,0.0,0.0,7290,1.0
2857,9079738443,60,2,0.0,0.0,7290,2.0
3774,9079738574,30,2,0.0,0.0,6240,1.0
3775,9079738574,60,2,0.0,0.0,6240,2.0
5153,9079740225,30,2,0.0,0.0,6120,1.0
5154,9079740225,60,2,0.0,0.0,6120,2.0


**Example: replay time and the game clock have different zero points**

```text
Match: 9079737909
First sampled in-progress state (state 5): tick 4260
Raw snapshot: tick 30
Incorrect game time: 30 / 30 = 1 second
Adjusted game time: (30 - 4260) / 30 = -141 seconds
```

The row is approximately **2 minutes 21 seconds before the chosen game-start point**, rather
than one second into the game. At tick `4260`, the adjusted clock is zero. Negative values preserve
pregame activity; they are not invalid timestamps to delete. This convention uses a once-per-second
state snapshot, so it does not independently establish the exact instant when movement starts.

**Fix:** subtract each match's game-start tick, preserve negative values, and compute duration
from final start/end game clocks. Stop if these required timing values are unavailable, rather
than fabricating a duration from the replay end.

In [38]:
timing = coverage.merge(starts, on="match_id", how="left").merge(
    last_rows(game, ["match_id"])[["match_id", "start_time", "end_time"]], on="match_id", how="left", validate="one_to_one")
timing["duration_s"] = (timing["end_time"] - timing["start_time"]).where(timing["end_time"] > 0)
timing_for_review = timing.loc[timing["game_start_tick"].isna() | timing["duration_s"].isna()
                               | timing["duration_s"].le(0).fillna(False)]
assert timing_for_review.empty, "Timing missing/invalid: inspect timing_for_review before proceeding."
def add_game_time(df):
    out = df.merge(starts, on="match_id", how="left", validate="many_to_one")
    out["game_time_s"] = (out["tick"] - out["game_start_tick"]) / 30.0
    assert out["game_time_s"].notna().all()
    return out.drop(columns="game_start_tick")
clean_game_clock = add_game_time(game)
assert clean_game_clock.merge(starts, on="match_id").loc[lambda d: d["tick"].eq(d["game_start_tick"]), "game_time_s"].eq(0).all()
display(clean_game_clock.groupby("match_id").head(2).head(12))
display(timing)
record("4. Game clock", "Validated on loaded matches", "Horn mapped to zero; pregame negative; duration from game clocks")

,match_id,tick,state,start_time,end_time,game_time_s
0,9079737909,30,2,0.0,0.0,-141.0
1,9079737909,60,2,0.0,0.0,-140.0
1508,9079737994,30,2,0.0,0.0,-204.0
1509,9079737994,60,2,0.0,0.0,-203.0
2856,9079738443,30,2,0.0,0.0,-242.0
2857,9079738443,60,2,0.0,0.0,-241.0
3774,9079738574,30,2,0.0,0.0,-207.0
3775,9079738574,60,2,0.0,0.0,-206.0
5153,9079740225,30,2,0.0,0.0,-203.0
5154,9079740225,60,2,0.0,0.0,-202.0


,match_id,game_start_tick,start_time,end_time,duration_s
0,9079737909,4260,154.733337,1376.800049,1222.06665
1,9079737994,6150,223.233337,1222.366699,999.133362
2,9079738443,7290,260.5,791.833374,531.333374
3,9079738574,6240,240.300018,1164.93335,924.633301
4,9079740225,6120,220.200012,416.766693,196.566681
5,9079740632,5580,211.800018,1384.000122,1172.200073
6,9079741152,6480,232.133347,962.000122,729.86676
7,9079743269,6030,237.600006,653.833374,416.233368
8,9079745340,6060,226.233353,855.833374,629.600037
9,9079746862,7110,258.300018,824.666687,566.366699


In [39]:
# Add game time to raw combat events
pregame_activity = add_game_time(combat_raw)
pregame_activity = pregame_activity.loc[
    (pregame_activity["match_id"] == 9079737909)
    & (pregame_activity["game_time_s"] < 0)
].copy()

# Resolve attacker, target, and ability/item names
for index_col, name_col in [
    ("attacker_name_idx", "attacker"),
    ("target_name_idx", "target"),
    ("inflictor_name_idx", "source"),
]:
    pregame_activity = resolve_name(
        pregame_activity, index_col, name_col
    )

# For purchases, value contains the purchased item's name index
pregame_activity = resolve_name(
    pregame_activity, "value", "purchase_item"
)
pregame_activity.loc[
    pregame_activity["type"] != "DOTA_COMBATLOG_PURCHASE",
    "purchase_item",
] = pd.NA

pregame_activity = pregame_activity.sort_values(
    ["tick"], kind="stable"
)

# Count the types of activity recorded
display(
    pregame_activity.groupby("type")
    .size()
    .rename("event_count")
    .reset_index()
)

# Show actual activity
display(
    pregame_activity[
        [
            "game_time_s",
            "tick",
            "type",
            "attacker",
            "target",
            "source",
            "value",
            "purchase_item",
        ]
    ].head(50)
)

,type,event_count
0,DOTA_COMBATLOG_GAME_STATE,8
1,DOTA_COMBATLOG_GOLD,10
2,DOTA_COMBATLOG_MODIFIER_ADD,18
3,DOTA_COMBATLOG_MODIFIER_STACK_EVENT,12
4,DOTA_COMBATLOG_PURCHASE,6


,game_time_s,tick,type,attacker,target,source,value,purchase_item
0,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,2,<NA>
1,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,12,<NA>
2,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,9,<NA>
3,-136.733333,158,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,3,<NA>
4,-106.733333,1058,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,8,<NA>
5,-90.733333,1538,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,10,<NA>
6,-90.666667,1540,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,4,<NA>
7,-90.6,1542,DOTA_COMBATLOG_GOLD,<NA>,npc_dota_hero_juggernaut,<NA>,600,<NA>
8,-90.6,1542,DOTA_COMBATLOG_MODIFIER_ADD,npc_dota_hero_legion_commander,npc_dota_hero_legion_commander,modifier_legion_commander_outfight_them,0,<NA>
9,-90.6,1542,DOTA_COMBATLOG_GOLD,<NA>,npc_dota_hero_legion_commander,<NA>,600,<NA>


9	-90.6	1542	DOTA_COMBATLOG_GOLD	<NA>	npc_dota_hero_legion_commander	<NA>	600	<NA>
10	-90.6	1542	DOTA_COMBATLOG_GOLD	<NA>	npc_dota_hero_tidehunter	<NA>	600	<NA>

this attributes gold to each player at the start of the game

49	-0.133333	4256	DOTA_COMBATLOG_PURCHASE	<NA>	npc_dota_hero_sniper	<NA>	25	item_tango

this is the first item bought, just before the game starts. assume the game start timer starts when players can walk out of fountain. 

## Issue 5. Damage needs name resolution, a source key, and aggregation

**Why transform:** the clean table's grain is attacker/target/source/tick. Several raw hits can
share that grain. Dropping repeated keys loses damage. Many ordinary attacks have no source
index; the draft contract uses `dota_unknown` so the clean key is non-null.

**Evidence cell:** show the repeated-key counts, a complete example group, and missing-source rows.
Missing event-specific fields are not a reason to drop every sparse combat-log row.

In [40]:
damage = combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_DAMAGE")
    & (true_mask(combat_raw["attacker_hero"]) | true_mask(combat_raw["target_hero"]))
    & false_mask(combat_raw["attacker_illusion"]) & false_mask(combat_raw["target_illusion"])].copy()
for index, label in [("attacker_name_idx", "attacker_name"), ("target_name_idx", "target_name"),
                     ("inflictor_name_idx", "inflictor_name")]:
    damage = resolve_name(damage, index, label)
damage_key = ["match_id", "tick", "attacker_name", "target_name", "inflictor_name"]
repeated_damage = duplicate_groups(damage, damage_key)
display(by_match(repeated_damage, "repeated_damage_groups"))
example_key = repeated_damage.sort_values("rows", ascending=False).head(1)[damage_key]
display(damage.merge(example_key, on=damage_key, how="inner")[damage_key + ["value"]])
print("not na inflictor name:")
display(
    damage.loc[
        ~damage["inflictor_name"].isna(),
        damage_key + ["value"],
    ]
)
missing_source = damage.loc[damage["inflictor_name_idx"].isna()]
display(by_match(missing_source, "missing_source_indexes"))
display(missing_source[["match_id", "tick", "attacker_name", "target_name", "inflictor_name_idx", "inflictor_name", "value"]].head(8))

,match_id,repeated_damage_groups
0,9079737909,256
1,9079737994,365
2,9079738443,743
3,9079738574,165
4,9079740225,135
5,9079740632,156
6,9079741152,160
7,9079743269,67
8,9079745340,90
9,9079746862,71


,match_id,tick,attacker_name,target_name,inflictor_name,value
0,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
1,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
2,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,18
3,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,18
4,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
5,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
6,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,19
7,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,19
8,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
9,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9


not na inflictor name:


,match_id,tick,attacker_name,target_name,inflictor_name,value
5,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,sven_great_cleave,42
6,9079737909,5360,npc_dota_hero_sven,npc_dota_hero_tidehunter,sven_great_cleave,36
7,9079737909,5360,npc_dota_hero_sven,npc_dota_hero_legion_commander,sven_great_cleave,27
8,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,sven_great_cleave,42
12,9079737909,5532,npc_dota_hero_sven,npc_dota_hero_tidehunter,sven_great_cleave,37
...,...,...,...,...,...,...
48947,9079746862,27088,npc_dota_hero_lina,npc_dota_creep_goodguys_ranged,lina_slow_burn,6
48952,9079746862,27114,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,46
48953,9079746862,27116,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,46
48957,9079746862,27144,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,13


,match_id,missing_source_indexes
0,9079737909,7505
1,9079737994,5630
2,9079738443,1489
3,9079738574,3383
4,9079740225,600
5,9079740632,4324
6,9079741152,2058
7,9079743269,1228
8,9079745340,1844
9,9079746862,1597


,match_id,tick,attacker_name,target_name,inflictor_name_idx,inflictor_name,value
0,9079737909,4954,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,49
1,9079737909,5216,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,63
2,9079737909,5256,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,44
3,9079737909,5308,npc_dota_hero_juggernaut,npc_dota_creep_badguys_melee,<NA>,<NA>,47
4,9079737909,5354,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,68
9,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,<NA>,<NA>,18
10,9079737909,5398,npc_dota_hero_drow_ranger,npc_dota_creep_goodguys_melee,<NA>,<NA>,7
11,9079737909,5516,npc_dota_hero_sniper,npc_dota_creep_badguys_ranged,<NA>,<NA>,54


**Example: repeated damage keys can represent multiple hits**

The evidence above contains this group:

```text
Match: 9079738574
Tick: 23834
Attacker: npc_dota_broodmother_spiderling
Target: npc_dota_hero_phantom_lancer
Source: <NA>
Raw rows with this key: 16
Recorded damage values include: 9, 18, and 19
```

These rows share the clean table's key, but that does not make them unwanted copies. Dropping
all but one would lose recorded damage. The clean representation sums all 16 values and sets
`hits = 16`. A shared unit name can also represent multiple same-named entities, so this aggregation
is about the agreed name-based table grain, not identifying one physical attacker.

The absent source index becomes `dota_unknown` under the contract. It means the replay did not
supply a source index; it does not mean damage was zero. A non-null index that fails to resolve
is a separate lookup problem and is retained for review.

**Fix:** apply `dota_unknown` to absent source indexes only; retain unresolved non-null indexes
and invalid required values for review. Sum damage and count hits per clean key. Check both
uniqueness and conservation of the valid input's total damage and hit count.

In [41]:
damage_ready = damage.copy()
damage_ready.loc[damage_ready["inflictor_name_idx"].isna(), "inflictor_name"] = "dota_unknown"
invalid_damage = damage_ready[damage_key].isna().any(axis=1) | damage_ready["value"].isna() | damage_ready["value"].lt(0).fillna(False)
damage_for_review = damage_ready.loc[invalid_damage].copy()
damage_ready = damage_ready.loc[~invalid_damage].copy()
clean_damage = damage_ready.groupby(damage_key, dropna=False).agg(
    damage=("value", lambda s: s.sum(min_count=1)), hits=("value", "size")).reset_index()
for field, label in [("attacker_name", "attacker_slot"), ("target_name", "target_slot")]:
    clean_damage = clean_damage.merge(clean_players[["match_id", "hero_name", "player_slot"]].rename(
        columns={"hero_name": field, "player_slot": label}), on=["match_id", field], how="left", validate="many_to_one")
clean_damage = add_game_time(clean_damage)
check_unique(clean_damage, damage_key)
assert clean_damage["damage"].sum() == damage_ready["value"].sum()
assert clean_damage["hits"].sum() == len(damage_ready)
assert len(damage_ready) + len(damage_for_review) == len(damage)
display(clean_damage.sort_values("hits", ascending=False).head(8))
print("Input eligible rows:", len(damage), "| Clean groups:", len(clean_damage), "| Review rows:", len(damage_for_review))
record("5. Damage aggregation/source", "Validated on loaded matches", f"{len(clean_damage)} groups; damage and hit totals conserved; {len(damage_for_review)} review rows")

,match_id,tick,attacker_name,target_name,inflictor_name,damage,hits,attacker_slot,target_slot,game_time_s
26072,9079738574,23868,npc_dota_hero_oracle,npc_dota_broodmother_spiderling,oracle_fortunes_end,1200,16,0,<NA>,587.6
26067,9079738574,23798,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,dota_unknown,242,16,<NA>,2,585.266667
26070,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,dota_unknown,221,16,<NA>,2,586.466667
25765,9079738574,21292,npc_dota_broodmother_spiderling,npc_dota_hero_arc_warden,dota_unknown,192,15,<NA>,1,501.733333
25757,9079738574,21256,npc_dota_broodmother_spiderling,npc_dota_hero_arc_warden,dota_unknown,190,15,<NA>,1,500.533333
11329,9079737994,18076,npc_dota_hero_bristleback,npc_dota_broodmother_spiderling,bristleback_quill_spray,570,12,4,<NA>,397.533333
11335,9079737994,18116,npc_dota_hero_sven,npc_dota_broodmother_spiderling,sven_great_cleave,564,12,3,<NA>,398.866667
26082,9079738574,23950,npc_dota_broodmother_spiderling,npc_dota_hero_oracle,dota_unknown,124,11,<NA>,0,590.333333


Input eligible rows: 48985 | Clean groups: 46043 | Review rows: 0


## Issue 6. Repeated purchases should become quantities

**Why transform:** buying the same item more than once at the same tick yields repeated clean
keys. Keeping one arbitrary row undercounts purchases. Names must first be resolved in the
correct match, then the buyer mapped to the roster.

**Evidence cell:** show repeated purchase groups and actual raw purchase rows from those groups.

In [42]:
purchases = resolve_name(resolve_name(combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_PURCHASE")],
                                      "target_name_idx", "buyer"), "value", "item_name")
purchase_key_raw = ["match_id", "tick", "buyer", "item_name"]
repeated_purchases = duplicate_groups(purchases, purchase_key_raw)
display(by_match(repeated_purchases, "repeated_purchase_groups"))
display(purchases.merge(repeated_purchases[purchase_key_raw], on=purchase_key_raw, how="inner")[
    purchase_key_raw + ["target_name_idx", "value"]].head(15))

,match_id,repeated_purchase_groups
0,9079737909,0
1,9079737994,3
2,9079738443,1
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,0
7,9079743269,0
8,9079745340,0
9,9079746862,1


,match_id,tick,buyer,item_name,target_name_idx,value
0,9079737994,8914,npc_dota_hero_meepo,item_wraith_band,18,117
1,9079737994,8914,npc_dota_hero_meepo,item_wraith_band,18,117
2,9079737994,9644,npc_dota_hero_broodmother,item_wraith_band,13,117
3,9079737994,9644,npc_dota_hero_broodmother,item_wraith_band,13,117
4,9079737994,10952,npc_dota_hero_kunkka,item_bracer,21,136
5,9079737994,10952,npc_dota_hero_kunkka,item_bracer,21,136
6,9079738443,12640,npc_dota_hero_skywrath_mage,item_null_talisman,10,133
7,9079738443,12640,npc_dota_hero_skywrath_mage,item_null_talisman,10,133
8,9079746862,23426,npc_dota_hero_nyx_assassin,item_branches,1,95
9,9079746862,23426,npc_dota_hero_nyx_assassin,item_branches,1,95


**Example: two purchases become one row with quantity two**

```text
Match: 9079737994
Tick: 8914
Buyer: npc_dota_hero_meepo
Item: item_wraith_band
Raw purchase rows with this key: 2
```

The contract counts identical purchase events at one tick. Its output is one row with
`quantity = 2`. Simply dropping repeated keys would keep only one purchase and undercount the
recorded events. The fix checks that the sum of output quantities equals the valid input row count;
it does not independently prove how many items the player owned afterward.

**Fix:** resolve the buyer to a player slot, count each group into `quantity`, and retain rows
with unresolved keys for review. The sum of quantities must equal the number of valid input rows.

In [43]:
purchases_ready = purchases.merge(clean_players[["match_id", "hero_name", "player_slot"]].rename(
    columns={"hero_name": "buyer"}), on=["match_id", "buyer"], how="left", validate="many_to_one")
purchase_key = ["match_id", "tick", "player_slot", "item_name"]
invalid_purchase = purchases_ready[purchase_key].isna().any(axis=1)
purchases_for_review = purchases_ready.loc[invalid_purchase].copy()
purchases_ready = purchases_ready.loc[~invalid_purchase].copy()
clean_purchases = grouped_counts(purchases_ready, purchase_key, "quantity")
clean_purchases = add_game_time(clean_purchases)
check_unique(clean_purchases, purchase_key)
assert clean_purchases["quantity"].sum() == len(purchases_ready)
assert len(purchases_ready) + len(purchases_for_review) == len(purchases)
display(clean_purchases.loc[clean_purchases["quantity"] > 1])
print("Raw purchase rows:", len(purchases), "| Clean rows:", len(clean_purchases), "| Review rows:", len(purchases_for_review))
record("6. Purchase quantity", "Validated on loaded matches", f"Quantities conserve {len(purchases_ready)} valid purchase rows")

,match_id,tick,player_slot,item_name,quantity,game_time_s
225,9079737994,8914,9,item_wraith_band,2,92.133333
232,9079737994,9644,6,item_wraith_band,2,116.466667
239,9079737994,10952,8,item_bracer,2,160.066667
402,9079738443,12640,5,item_null_talisman,2,178.333333
1341,9079746862,23426,0,item_branches,2,543.866667


Raw purchase rows: 1365 | Clean rows: 1360 | Review rows: 0


## Issue 7. Combat-log participants do not establish scoreboard K/D/A credit

**Why change:** the direct attacker and `assist_players` describe combat participants,
not the complete credit ledger. Towers and summons can land the last hit while a player
gets kill credit. Illusion attackers must not hide real deaths, reincarnation events may
not count as deaths, and listed participants can include killers or omit credited assists.

**Evidence:** reproduce the old cleaner's counting below: reject illusion attackers,
drop conflicting death keys, map direct attackers to heroes, and count listed assistants
except the direct killer. Compare those counts with each player's final raw scoreboard.
These are legacy counts for comparison, not the new clean tables.


In [44]:
death_key = ["match_id", "tick", "victim_slot"]
assist_key = [*death_key, "assist_slot"]
deaths_named = resolve_name(resolve_name(resolve_name(
    combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_DEATH")],
    "attacker_name_idx", "killer_name"), "target_name_idx", "victim_name"),
    "damage_source_name_idx", "source_name")
death_events = deaths_named.loc[
    true_mask(deaths_named["target_hero"]) & false_mask(deaths_named["target_illusion"])
].copy()
for name, slot in [("victim_name", "victim_slot"), ("killer_name", "killer_slot")]:
    death_events = death_events.merge(
        clean_players[["match_id", "hero_name", "player_slot"]].rename(
            columns={"hero_name": name, "player_slot": slot}),
        on=["match_id", name], how="left", validate="many_to_one")
assert death_events["victim_slot"].notna().all(), "Unresolved real-hero victim"

legacy_deaths = death_events.loc[false_mask(death_events["attacker_illusion"])].drop_duplicates().copy()
legacy_conflict_keys = duplicate_groups(legacy_deaths, death_key)[death_key]
legacy_deaths = legacy_deaths.merge(legacy_conflict_keys.assign(conflict=True),
    on=death_key, how="left", validate="many_to_one")
legacy_deaths = legacy_deaths.loc[legacy_deaths["conflict"].isna()].drop(columns="conflict")
legacy_assists = legacy_deaths[[*death_key, "killer_slot", "assist_players"]].copy()
legacy_assists["token"] = legacy_assists["assist_players"].fillna("").str.split(";")
legacy_assists = legacy_assists.explode("token", ignore_index=True)
legacy_assists["token"] = legacy_assists["token"].str.strip()
legacy_assists = legacy_assists.loc[legacy_assists["token"].str.fullmatch(r"[0-9]+").fillna(False)].copy()
legacy_assists["assist_slot"] = pd.to_numeric(legacy_assists["token"]).astype("Int64")
legacy_assists = legacy_assists.loc[legacy_assists["assist_slot"].between(0, 9)
    & (legacy_assists["killer_slot"].isna()
       | legacy_assists["assist_slot"].ne(legacy_assists["killer_slot"]).fillna(False))].drop_duplicates(assist_key)

stats = ["kills", "deaths", "assists"]
last_stats = last_rows(resource_mapped, ["match_id", "player_slot"])[[
    "match_id", "player_slot", *[f"m_vecPlayerTeamData.m_i{s.title()}" for s in stats]
]].rename(columns={f"m_vecPlayerTeamData.m_i{s.title()}": f"scoreboard_{s}" for s in stats})

def add_credit_count(frame, events, slot_column, label):
    counts = grouped_counts(events.dropna(subset=[slot_column]), ["match_id", slot_column], label)
    counts = counts.rename(columns={slot_column: "player_slot"})
    result = frame.merge(counts, on=["match_id", "player_slot"], how="left", validate="one_to_one")
    result[label] = result[label].fillna(0).astype("Int64")
    return result

legacy_comparison = last_stats.copy()
for events, slot, stat in [(legacy_deaths, "killer_slot", "kills"),
                           (legacy_deaths, "victim_slot", "deaths"),
                           (legacy_assists, "assist_slot", "assists")]:
    legacy_comparison = add_credit_count(legacy_comparison, events, slot, f"legacy_{stat}")
legacy_comparison["legacy_mismatch"] = False
for stat in stats:
    legacy_comparison["legacy_mismatch"] |= legacy_comparison[f"legacy_{stat}"].ne(
        legacy_comparison[f"scoreboard_{stat}"]).fillna(True)
display(by_match(legacy_comparison.loc[legacy_comparison["legacy_mismatch"]], "legacy_mismatching_players"))
display(legacy_comparison.loc[legacy_comparison["legacy_mismatch"]].sort_values(["match_id", "player_slot"]))


,match_id,legacy_mismatching_players
0,9079737909,1
1,9079737994,6
2,9079738443,0
3,9079738574,4
4,9079740225,0
5,9079740632,5
6,9079741152,1
7,9079743269,1
8,9079745340,2
9,9079746862,1


,match_id,player_slot,scoreboard_kills,scoreboard_deaths,scoreboard_assists,legacy_kills,legacy_deaths,legacy_assists,legacy_mismatch
4,9079737909,4,2,8,0,0,8,2,True
13,9079737994,3,3,10,1,2,10,3,True
14,9079737994,4,4,6,2,3,6,1,True
16,9079737994,6,8,1,4,7,1,5,True
17,9079737994,7,5,1,13,4,1,14,True
18,9079737994,8,6,4,12,5,4,13,True
19,9079737994,9,16,2,9,14,0,10,True
30,9079738574,0,1,8,2,1,8,1,True
31,9079738574,1,1,5,0,0,5,1,True
35,9079738574,5,6,2,10,6,2,7,True


**Example: last hitter and credited killer differ**

In match `9079737909`, Sniper at slot `4` has two scoreboard kills, while direct
attacker-name counting finds zero. The last hitters are towers. Removing the direct
killer from an assist list cannot establish the correct player credit in this case.

**New method:** extract raw counters from `CDOTA_PlayerResource` every tick into
`kda_changes`, retaining the initial values and every team/slot/hero/K/D/A change.
Calculate counter increases in clean. Match a death to a rise in the victim's death
counter within the next four ticks, then assign kill/assist credit using the counter
increases and source/participant constraints. Block non-unique or inconsistent answers.
Final totals must still equal the independently sampled scoreboard exactly.

The next cells inspect that input, then load and independently check the saved output
of the actual Spark solver. They do not invent a second attribution algorithm in pandas.


In [45]:
kda = batch["kda_changes"].rename(columns={
    "m_vecPlayerData.m_iPlayerTeam": "team",
    "m_vecPlayerTeamData.m_iTeamSlot": "team_slot",
    "m_vecPlayerTeamData.m_nSelectedHeroID": "hero_id",
    **{f"m_vecPlayerTeamData.m_i{s.title()}": s for s in stats},
}).copy()
assigned_kda = kda.loc[kda["team"].isin([2, 3])].copy()
assert assigned_kda["team_slot"].between(0, 4).fillna(False).all()
index_mapping = assigned_kda[["match_id", "index", "team", "team_slot"]].drop_duplicates()
index_mapping["player_slot"] = ((index_mapping["team"] - 2) * 5 + index_mapping["team_slot"]).astype("Int64")
check_unique(index_mapping, ["match_id", "index"])
check_unique(index_mapping, ["match_id", "player_slot"])
index_mapping = index_mapping.merge(roster, on=["match_id", "player_slot"], how="left", validate="one_to_one")
assert index_mapping["team"].eq(index_mapping["header_team"]).fillna(False).all()

# A later stable index mapping also identifies initial rows before team assignment.
kda_history = kda.merge(index_mapping[["match_id", "index", "player_slot"]],
    on=["match_id", "index"], how="inner", validate="many_to_one")
kda_history = kda_history.sort_values(["match_id", "player_slot", "tick"]).copy()
check_unique(kda_history, ["match_id", "player_slot", "tick"])
assert kda_history[stats].notna().all(axis=None)
first_counter_row = kda_history.groupby(["match_id", "player_slot"]).cumcount().eq(0)
assert kda_history.loc[first_counter_row, stats].eq(0).all(axis=None), "Missing initial zero counters"
deltas = kda_history.groupby(["match_id", "player_slot"])[stats].diff()
deltas.loc[first_counter_row, :] = 0
assert deltas.ge(0).all(axis=None), "A counter decreased"
for stat in stats:
    kda_history[f"delta_{stat}"] = deltas[stat].astype("Int64")

counter_totals = kda_history.groupby(["match_id", "player_slot"])[
    [f"delta_{s}" for s in stats]].sum().reset_index().rename(
        columns={f"delta_{s}": f"counter_{s}" for s in stats})
counter_check = last_stats.merge(counter_totals, on=["match_id", "player_slot"],
    how="outer", validate="one_to_one", indicator=True)
assert counter_check["_merge"].eq("both").all()
for stat in stats:
    assert counter_check[f"counter_{stat}"].eq(counter_check[f"scoreboard_{stat}"]).all()
counter_changes = kda_history.loc[kda_history[[f"delta_{s}" for s in stats]].gt(0).any(axis=1)]
display(by_match(kda_history, "per_tick_change_rows"))
display(counter_changes.loc[counter_changes["match_id"].eq(MATCH_ID),
    ["match_id", "tick", "player_slot", *stats, *[f"delta_{s}" for s in stats]]].head(20))
print("Player counter histories matching the final raw scoreboard:", len(counter_check))


,match_id,per_tick_change_rows
0,9079737909,117
1,9079737994,196
2,9079738443,43
3,9079738574,114
4,9079740225,49
5,9079740632,247
6,9079741152,95
7,9079743269,66
8,9079745340,58
9,9079746862,52


,match_id,tick,player_slot,kills,deaths,assists,delta_kills,delta_deaths,delta_assists
28,9079737909,8452,0,0,1,0,0,1,0
39,9079737909,10688,0,0,2,0,0,1,0
54,9079737909,12920,0,0,2,1,0,0,1
59,9079737909,13194,0,0,2,2,0,0,1
70,9079737909,15276,0,0,3,2,0,1,0
73,9079737909,15524,0,0,3,3,0,0,1
87,9079737909,17680,0,0,3,4,0,0,1
95,9079737909,18000,0,0,3,5,0,0,1
107,9079737909,20050,0,1,3,5,1,0,0
31,9079737909,8456,1,0,0,1,0,0,1


Player counter histories matching the final raw scoreboard: 100


**Read the completed counter-based clean output.**

Set `CLEAN_KDA` in setup to change the output folder. Every loaded match needs a matching
report with `credit_method = per_tick_counters`, the new contract version, no errors,
`complete = true`, and a completion marker. Preview or failed outputs are not accepted
as completed data. This reads the per-match reports directly, not the saved batch summary.

If output is missing, run `spark_jobs/clean_replays.py` for the match first using the
[Docker instructions](../docs/running-kda-clean-job.md). This notebook does not run the job.


In [46]:
clean_reports = {}
run_rows = []
for match_id in COMPLETE_MATCH_IDS:
    report_path = CLEAN_KDA / "_review" / f"match_id={match_id}" / "report.json"
    if not report_path.is_file():
        raise FileNotFoundError(f"Run the counter-based Spark cleaner first: {report_path}")
    report = json.loads(report_path.read_text())
    marker = CLEAN_KDA / "_complete" / f"match_id={match_id}"
    ready = (report.get("match_id") == match_id
        and report.get("credit_method") == "per_tick_counters"
        and report.get("contract_version") == "kda-credit-v1"
        and report.get("complete") is True and report.get("preview") is False
        and report.get("errors") == [] and marker.is_dir()
        and any(p.is_file() for p in marker.rglob("*")))
    clean_reports[match_id] = report
    run_rows.append({"match_id": match_id, "complete": ready, "errors": report.get("errors")})
clean_run_status = pd.DataFrame(run_rows)
display(clean_run_status)
assert clean_run_status["complete"].all(), "Inspect failed/preview reports before checking clean credit"

credit_dtypes = {
    "deaths": {"match_id": "Int64", "tick": "Int64", "game_time_s": "Float64",
               "victim_slot": "Int64", "killer_slot": "Int64", "killer_name": "string",
               "credited_killer_slot": "Int64", "link_status": "string"},
    "death_assists": {"match_id": "Int64", "tick": "Int64", "victim_slot": "Int64",
                      "assist_slot": "Int64", "same_side": "boolean"},
}

def read_clean_credit(table, match_id):
    folder = CLEAN_KDA / table / f"match_id={match_id}"
    files = sorted(folder.glob("*.parquet"))
    expected = clean_reports[match_id]["table_rows"][table]
    if files:
        frame = pd.concat([pq.ParquetFile(f).read().to_pandas().convert_dtypes()
                           for f in files], ignore_index=True)
        if "match_id" in frame:
            assert frame["match_id"].eq(match_id).all(), f"Wrong match_id in {folder}"
        frame["match_id"] = pd.Series(match_id, index=frame.index, dtype="Int64")
    else:
        assert expected == 0, f"Missing published credit table: {folder}"
        frame = pd.DataFrame({name: pd.Series(dtype=dtype)
                              for name, dtype in credit_dtypes[table].items()})
    assert set(credit_dtypes[table]).issubset(frame.columns), f"Old credit schema: {folder}"
    assert len(frame) == expected, f"Report row count differs from saved files: {folder}"
    return frame.astype(credit_dtypes[table])

clean_deaths = pd.concat([read_clean_credit("deaths", m) for m in COMPLETE_MATCH_IDS], ignore_index=True)
clean_death_assists = pd.concat([read_clean_credit("death_assists", m) for m in COMPLETE_MATCH_IDS], ignore_index=True)
print("Credited deaths:", len(clean_deaths), "| Credited assists:", len(clean_death_assists))


,match_id,complete,errors
0,9079737909,True,[]
1,9079737994,True,[]
2,9079738443,True,[]
3,9079738574,True,[]
4,9079740225,True,[]
5,9079740632,True,[]
6,9079741152,True,[]
7,9079743269,True,[]
8,9079745340,True,[]
9,9079746862,True,[]


Credited deaths: 277 | Credited assists: 243


In [47]:
# Recount published credit independently. Kills use CREDITED killer, not last hitter.
kda_comparison = last_stats.copy()
for events, slot, stat in [(clean_deaths, "credited_killer_slot", "kills"),
                           (clean_deaths, "victim_slot", "deaths"),
                           (clean_death_assists, "assist_slot", "assists")]:
    assert events[slot].dropna().isin(range(10)).all()
    kda_comparison = add_credit_count(kda_comparison, events, slot, f"credited_{stat}")
kda_comparison = kda_comparison.merge(
    legacy_comparison[["match_id", "player_slot", "legacy_mismatch"]],
    on=["match_id", "player_slot"], how="left", validate="one_to_one")
for stat in stats:
    kda_comparison[f"{stat}_mismatch"] = kda_comparison[f"credited_{stat}"].ne(
        kda_comparison[f"scoreboard_{stat}"]).fillna(True)
kda_comparison["new_mismatch"] = kda_comparison[[f"{s}_mismatch" for s in stats]].any(axis=1)
kda_disagreements = kda_comparison.loc[kda_comparison["new_mismatch"]].copy()
kda_summary = kda_comparison.groupby("match_id").agg(
    players_checked=("player_slot", "size"), legacy_mismatching_players=("legacy_mismatch", "sum"),
    new_mismatching_players=("new_mismatch", "sum")).reset_index()
display(kda_summary)
display(kda_comparison.loc[kda_comparison["match_id"].eq(MATCH_ID),
    ["match_id", "player_slot", *[c for s in stats for c in (f"scoreboard_{s}", f"credited_{s}")]]])
assert kda_disagreements.empty, "Published K/D/A differs from the independent raw scoreboard"
assert kda_summary["players_checked"].eq(10).all()
print(f"Checked {len(kda_comparison)} players / {len(kda_comparison) * 3} K/D/A values; "
      f"{len(kda_disagreements)} mismatching players")
record("7. Counter-based K/D/A credit", "Validated on loaded completed outputs",
       f"{len(kda_comparison)} player totals checked; {len(kda_disagreements)} K/D/A mismatches")


,match_id,players_checked,legacy_mismatching_players,new_mismatching_players
0,9079737909,10,1,0
1,9079737994,10,6,0
2,9079738443,10,0,0
3,9079738574,10,4,0
4,9079740225,10,0,0
5,9079740632,10,5,0
6,9079741152,10,1,0
7,9079743269,10,1,0
8,9079745340,10,2,0
9,9079746862,10,1,0


,match_id,player_slot,scoreboard_kills,credited_kills,scoreboard_deaths,credited_deaths,scoreboard_assists,credited_assists
0,9079737909,0,1,1,3,3,5,5
1,9079737909,1,3,3,3,3,4,4
2,9079737909,2,4,4,2,2,3,3
3,9079737909,3,2,2,3,3,4,4
4,9079737909,4,2,2,8,8,0,0
5,9079737909,5,1,1,4,4,5,5
6,9079737909,6,2,2,5,5,2,2
7,9079737909,7,2,2,3,3,2,2
8,9079737909,8,6,6,2,2,0,0
9,9079737909,9,5,5,4,4,1,1


Checked 100 players / 300 K/D/A values; 0 mismatching players


## Issue 8. Death records, last hitters, and credited players have different meanings

**Why change:** a real hero death can have multiple combat records (for example Meepo),
while an Aegis or reincarnation event can have no scoreboard death. The immediate last
hitter also need not be the credited player. Quarantining every repeated key and every
non-player last hit leaves the output incomplete.

**Evidence:** inspect repeated raw real-hero death keys, then compare the published
last hitter with `credited_killer_slot`. Unlike the legacy comparison above, raw evidence
here includes illusion attackers. Illusion victims are excluded.


In [48]:
repeated_death_groups = duplicate_groups(death_events, death_key)
repeated_death_records = death_events.merge(repeated_death_groups[death_key],
    on=death_key, how="inner", validate="many_to_one")
display(by_match(repeated_death_groups, "repeated_raw_death_keys"))
evidence_columns = death_key + ["killer_name", "killer_slot", "victim_name", "source_name",
                               "attacker_illusion", "assist_players", "target_self", "will_reincarnate"]
display(repeated_death_records[evidence_columns].sort_values(death_key).head(20))
last_hitter_differs = clean_deaths.loc[clean_deaths["credited_killer_slot"].notna()
    & (clean_deaths["killer_slot"].isna()
       | clean_deaths["killer_slot"].ne(clean_deaths["credited_killer_slot"]).fillna(False))].copy()
display(last_hitter_differs[[*death_key, "killer_name", "killer_slot", "credited_killer_slot", "link_status"]])
raw_death_status = death_events[death_key].drop_duplicates().merge(
    clean_deaths[death_key].assign(counted_scoreboard_death=True),
    on=death_key, how="left", validate="one_to_one")
uncounted_raw_deaths = raw_death_status.loc[raw_death_status["counted_scoreboard_death"].isna(), death_key]
display(by_match(uncounted_raw_deaths, "raw_death_keys_not_counted"))


,match_id,repeated_raw_death_keys
0,9079737909,0
1,9079737994,2
2,9079738443,0
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,0
7,9079743269,0
8,9079745340,0
9,9079746862,0


,match_id,tick,victim_slot,killer_name,killer_slot,victim_name,source_name,attacker_illusion,assist_players,target_self,will_reincarnate
0,9079737994,25036,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,npc_dota_hero_meepo,False,<NA>,True,False
1,9079737994,25036,9,npc_dota_hero_sven,3,npc_dota_hero_meepo,npc_dota_hero_sven,False,4;3,False,False
2,9079737994,33150,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,npc_dota_hero_meepo,False,4,True,False
3,9079737994,33150,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,npc_dota_hero_meepo,False,<NA>,False,False
4,9079737994,33150,9,npc_dota_hero_bristleback,4,npc_dota_hero_meepo,npc_dota_hero_bristleback,False,4,False,False


,match_id,tick,victim_slot,killer_name,killer_slot,credited_killer_slot,link_status
23,9079737909,17682,8,npc_dota_goodguys_tower1_mid,<NA>,4,exact
29,9079737909,19898,8,npc_dota_goodguys_tower1_mid,<NA>,4,exact
40,9079737994,6852,1,npc_dota_badguys_tower1_mid,<NA>,9,exact
49,9079737994,10868,0,npc_dota_badguys_tower1_mid,<NA>,7,exact
56,9079737994,13730,0,npc_dota_badguys_tower1_mid,<NA>,9,exact
59,9079737994,14544,2,npc_dota_creep_badguys_ranged,<NA>,6,exact
80,9079737994,23150,0,npc_dota_badguys_tower1_mid,<NA>,8,exact
118,9079738574,18500,6,npc_dota_goodguys_tower2_mid,<NA>,1,exact
127,9079738574,23874,2,npc_dota_broodmother_spiderling,<NA>,6,exact
131,9079738574,28800,3,npc_dota_broodmother_spiderling,<NA>,6,exact


,match_id,raw_death_keys_not_counted
0,9079737909,0
1,9079737994,0
2,9079738443,0
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,0
7,9079743269,0
8,9079745340,0
9,9079746862,0


**Fix used by the Spark job:** merge a same-tick/same-victim self-source duplicate with
its consistent non-self record. Block sources that still conflict. Count a death only
when its victim's death counter rises in the allowed window; every counter increment
must be accounted for. A reincarnation flag alone does not decide whether a death counts.

`killer_slot` and `killer_name` retain the direct last hitter. `credited_killer_slot`
identifies scoreboard kill credit. `death_assists` records counter-established assists,
including unusual same-team credit with `same_side = true`.

For example, Meepo's self-source record and Sven's non-self record at tick `25036` of
match `9079737994` no longer require discarding the whole death. The counter rise and
consistent source establish whether it is one scoreboard death.

**Uncertainty remains explicit:** `participant_assumed` means a link depends on participant
constraints; `same_side_possible` means allowing same-team credit provides another answer.
Exact final totals do not prove every event link. Review notes and these labels are part
of the result, even when the match passes.


In [49]:
check_unique(clean_deaths, death_key)
check_unique(clean_death_assists, assist_key)
assert clean_deaths["link_status"].isin(["exact", "participant_assumed", "same_side_possible"]).all()
assert clean_death_assists["same_side"].notna().all()

# Independently inspect allowed death-counter timing. These are candidate links;
# this timing join alone is not a second proof of unique attribution.
death_increments = kda_history.loc[kda_history["delta_deaths"].gt(0),
    ["match_id", "player_slot", "tick", "delta_deaths"]].rename(
        columns={"player_slot": "victim_slot", "tick": "counter_tick"})
windows = pd.DataFrame([{"match_id": m, "window_ticks": clean_reports[m]["window_ticks"]}
                        for m in COMPLETE_MATCH_IDS])
counter_windows = clean_deaths[death_key].merge(death_increments,
    on=["match_id", "victim_slot"], how="left", validate="many_to_many").merge(
        windows, on="match_id", how="left", validate="many_to_one")
counter_windows["tick_gap"] = counter_windows["counter_tick"] - counter_windows["tick"]
counter_windows = counter_windows.loc[counter_windows["tick_gap"].ge(0)
    & counter_windows["tick_gap"].le(counter_windows["window_ticks"])].copy()
assert len(counter_windows[death_key].drop_duplicates()) == len(clean_deaths)
display(counter_windows.loc[counter_windows["match_id"].eq(MATCH_ID)].head(12))

# Check clock values and same-side flags against the independent roster.
clock_check = clean_deaths.merge(starts, on="match_id", how="left", validate="many_to_one")
assert (clock_check["game_time_s"] - (clock_check["tick"] - clock_check["game_start_tick"]) / 30).abs().lt(1e-9).all()
side_check = clean_death_assists.copy()
for slot, label in [("victim_slot", "victim_team"), ("assist_slot", "assist_team")]:
    side_check = side_check.merge(clean_players[["match_id", "player_slot", "team"]].rename(
        columns={"player_slot": slot, "team": label}),
        on=["match_id", slot], how="left", validate="many_to_one")
assert side_check[["victim_team", "assist_team"]].notna().all(axis=None)
assert side_check["same_side"].eq(side_check["victim_team"].eq(side_check["assist_team"])).all()
assumed_death_links = clean_deaths.loc[clean_deaths["link_status"].ne("exact")].copy()
same_side_assists = side_check.loc[side_check["same_side"]].copy()
review_parts = []
for match_id in COMPLETE_MATCH_IDS:
    folder = CLEAN_KDA / "_review" / f"match_id={match_id}" / "kda_credit_review"
    review_parts.extend(pq.ParquetFile(p).read().to_pandas().convert_dtypes()
                        for p in sorted(folder.glob("*.parquet")))
kda_review_notes = (pd.concat(review_parts, ignore_index=True) if review_parts
                    else pd.DataFrame(columns=["match_id", "tick", "victim_slot", "reason"]))
display(assumed_death_links)
display(same_side_assists)
if not kda_review_notes.empty:
    display(grouped_counts(kda_review_notes, ["match_id", "reason"]))
print("Credited deaths:", len(clean_deaths), "| Credited assists:", len(clean_death_assists))
print("Assumption-labelled death links:", len(assumed_death_links), "| Same-side assists:", len(same_side_assists))
record("8. Death records and attribution", "Validated totals; link labels retained",
       f"{len(clean_deaths)} deaths, {len(clean_death_assists)} assists; "
       f"{len(assumed_death_links)} assumption-labelled links, {len(same_side_assists)} same-side assists")


,match_id,tick,victim_slot,counter_tick,delta_deaths,window_ticks,tick_gap
0,9079737909,8224,4,8226,1,4,2
8,9079737909,8450,0,8452,1,4,2
11,9079737909,8454,7,8456,1,4,2
14,9079737909,8518,6,8520,1,4,2
20,9079737909,10674,4,10676,1,4,2
28,9079737909,10686,0,10688,1,4,2
30,9079737909,10916,3,10918,1,4,2
34,9079737909,10930,6,10932,1,4,2
38,9079737909,11004,2,11006,1,4,2
40,9079737909,11388,1,11390,1,4,2


,tick,game_time_s,victim_slot,killer_slot,killer_name,credited_killer_slot,link_status,match_id
239,25752,642.4,2,7,npc_dota_hero_juggernaut,7,same_side_possible,9079741152
240,25752,642.4,7,2,npc_dota_hero_axe,2,same_side_possible,9079741152


,tick,victim_slot,assist_slot,same_side,match_id,victim_team,assist_team
102,24038,0,0,True,9079738574,2,2


,match_id,reason,rows
0,9079737994,merged_death_records,2


Credited deaths: 277 | Credited assists: 243
Assumption-labelled death links: 2 | Same-side assists: 1


## What has been checked, and what remains labelled

Issues 1-6 check the in-memory roster mapping, hero selection, game clock, damage
aggregation and purchase quantities. Their raw inputs remain unchanged.

Issues 7-8 read completed outputs from `data/output/clean_kda` and independently check:

- Every initial player counter is zero, increases never go backwards, and summed
  increases equal the final raw scoreboard.
- Published kills use `credited_killer_slot`; published K/D/A totals exactly match
  all loaded players' final raw scoreboard values.
- Death/assist keys are unique, counted deaths have a candidate counter rise in the
  configured window, and clock values and same-side flags agree with raw inputs.
- Reports, completion markers, schemas and saved credit-table row counts agree.

`clean_deaths` and `clean_death_assists` are loaded published output, not provisional
participant-list guesses. `assumed_death_links`, `same_side_assists`, and
`kda_review_notes` expose labels and review evidence. Matching totals do not independently
prove each assist-to-death link, and these checks are against replay counters rather
than an external Steam reconciliation.

This notebook does not regenerate outputs or update the warehouse. The clean contract, warehouse migration and loader include these credit columns.
The dbt models also use credited killers and retain all assists for K/D/A checks;
participation uses only eligible assists on the player's team's credited kills.


In [50]:
status = pd.DataFrame(results)
with pd.option_context("display.max_colwidth", None):
    display(status)
review_counts = pd.DataFrame([
    ("hero_ids_for_review", len(hero_ids_for_review)),
    ("invalid_resource_rows", len(invalid_resource_rows)),
    ("hero_selection_for_review", len(hero_selection_for_review)),
    ("timing_for_review", len(timing_for_review)),
    ("damage_for_review", len(damage_for_review)),
    ("purchases_for_review", len(purchases_for_review)),
    ("kda_disagreements", len(kda_disagreements)),
    ("assumed_death_links", len(assumed_death_links)),
    ("same_side_assists", len(same_side_assists)),
    ("kda_review_notes", len(kda_review_notes)),
], columns=["review_dataframe", "rows"])
display(review_counts)
display(kda_summary)


,issue,status,detail
0,1. Hero player IDs,Validated on loaded matches,0 hero rows retained for review
1,2. Non-roster resource rows,Filtered with rows retained for review,733 rows excluded from player stats
2,3. Stable hero assignment,Sample rule demonstrated,Final snapshot selected; 0 missing final IDs
3,4. Game clock,Validated on loaded matches,Horn mapped to zero; pregame negative; duration from game clocks
4,5. Damage aggregation/source,Validated on loaded matches,46043 groups; damage and hit totals conserved; 0 review rows
5,6. Purchase quantity,Validated on loaded matches,Quantities conserve 1365 valid purchase rows
6,7. Counter-based K/D/A credit,Validated on loaded completed outputs,100 player totals checked; 0 K/D/A mismatches
7,8. Death records and attribution,Validated totals; link labels retained,"277 deaths, 243 assists; 2 assumption-labelled links, 1 same-side assists"


,review_dataframe,rows
0,hero_ids_for_review,0
1,invalid_resource_rows,733
2,hero_selection_for_review,0
3,timing_for_review,0
4,damage_for_review,0
5,purchases_for_review,0
6,kda_disagreements,0
7,assumed_death_links,2
8,same_side_assists,1
9,kda_review_notes,2


,match_id,players_checked,legacy_mismatching_players,new_mismatching_players
0,9079737909,10,1,0
1,9079737994,10,6,0
2,9079738443,10,0,0
3,9079738574,10,4,0
4,9079740225,10,0,0
5,9079740632,10,5,0
6,9079741152,10,1,0
7,9079743269,10,1,0
8,9079745340,10,2,0
9,9079746862,10,1,0
